# 3. Ablations, read against the noise

An ablation delta is meaningless without knowing how far two runs of the *same* configuration drift apart. The seed study is therefore run first and every delta is divided by `sqrt(2) * sd` before it is interpreted.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd, torch
torch.set_num_threads(2)
pd.set_option("display.width", 200)
import matplotlib.pyplot as plt
tables = pathlib.Path('../results/tables')
var = pd.read_csv(tables / 'seed_variance.csv') if (tables / 'seed_variance.csv').exists() else None
var.round(4) if var is not None else 'run `make ablate` first'

In [ ]:
abl = pd.read_csv(tables / 'ablation_components.csv') if (tables / 'ablation_components.csv').exists() else None
cols = ['variant', 'change', 'spearman', 'spearman_delta', 'spearman_ratio_to_noise', 'spearman_verdict']
abl[[c for c in cols if c in abl.columns]].round(4) if abl is not None else None

## The normalisation choice is load-bearing

The first version of this model used GroupNorm everywhere, for batch-size independence. It could not fit its own training set. GroupNorm removes each sample's per-channel scale at every layer, and the readout is a global average pool over (T, V) -- which reads precisely that scale. This is measured below, not asserted.

In [ ]:
from saqa.config import load_config
from saqa.pipelines import make_splits, run_single
from saqa.metrics import spearman
from saqa.engine import predict
cfg = load_config('../configs/base.yaml', ['data.num_sequences=500', 'optim.epochs=10', 'run.out_dir=../results/runs_nb'])
sp = make_splits(cfg)
for norm in ('batch', 'group'):
    c = load_config('../configs/base.yaml', ['data.num_sequences=500', 'optim.epochs=10', f'model.norm={norm}', 'run.out_dir=../results/runs_nb'])
    r = run_single(c, name=f'nb_norm_{norm}', splits=sp, save=False, verbose=False)
    tr = spearman(sp.train.quality, predict(r.model, sp.train.coords)['score'])
    print(f'{norm:6s} train rho {tr:+.4f}   test rho {r.metrics["spearman"]:+.4f}')

## Splitting regimes: the leakage is measured, not assumed

A random split lets the same subject and the same defect combination appear on both sides. The gap between it and the cross-subject split is the leakage estimate.

In [ ]:
p = tables / 'split_comparison.csv'
pd.read_csv(p)[['split', 'spearman', 'kendall_tau', 'relative_l2', 'mae']].round(4) if p.exists() else 'run `make splits` first'